# イテレータを並列に処理するにはZipを使う

Python では、関係した多数のオブジェクトのリストを扱うことがよくあります。

リスト内包表記を使用すると、元となるリストを簡単に取得し、式を適用して新たなリストを取得できます。（項目27）

In [5]:
names = ['Cecilia', 'Lisa', 'Marie']
counts = [len(n) for n in names]
print(counts)

[7, 4, 5]


両方のリスト（names, counts）を並列に処理したければ、以下のようにします。

In [6]:
longest_name = None
max_count = 0

for i in range(len(names)):
  count = counts[i]
  if count > max_count:
    longest_name = names[i]
    max_count = count

print(longest_name)

Cecilia


このループ文は、全体の見た目がすっきりしません。

このようなコードをもっと明確にするために、Python には組み込み関数 zip があります。

zip は、2つ以上のイテレータを遅延評価ジェネレータでラップします。

zip ジェネレータは、各イテレータからの次の値のタプルを yield します。

In [7]:
longest_name = None
max_count = 0

for taple in zip(names, counts):
  print(taple)

('Cecilia', 7)
('Lisa', 4)
('Marie', 5)


これらのタプルは、for文の内側で直接アンパックできます。

In [8]:
longest_name = None
max_count = 0

for name, count in zip(names, counts):
  if count > max_count:
    longest_name = name
    max_count = count

print(longest_name)

Cecilia


zip はラップしているイテレータで要素を1つずつ処理します。したがって、無限に長い入力でもメモリを使いすぎてクラッシュする危険性はありません。（補足にて説明）

しかし、入力イテレータの長さが異なる場合には、zip の振る舞いを注意する必要があります。

In [ ]:
names.append('Rosalind')
# append 後に counts の更新をわすれると最短のリストまでしか処理されない
# Rosalind が表示されない
for name, count in zip(names, counts):
  print(f'{name}: {count}')

Cecilia: 7
Lisa: 4
Marie: 5


すべてを処理せずに終わるのは好ましくないときが大抵である

Zip の対象となるリストの長さが同じである確証を持てないのであれば、 zip_longest 関数を使うことを検討する

In [10]:
import itertools

for name, count in itertools.zip_longest(names, counts):
  print(f'{name}: {count}')

Cecilia: 7
Lisa: 4
Marie: 5
Rosalind: None


In [11]:
# counts の更新をした場合
counts = [len(n) for n in names]
for name, count in itertools.zip_longest(names, counts):
  print(f'{name}: {count}')

Cecilia: 7
Lisa: 4
Marie: 5
Rosalind: 8


## 覚えておくこと

- 組み込み関数 zip は、複数のイテレータを並列に処理するのに使う。
- zip はタプルを生成する遅延評価ジェネレータなので、無限に長い入力にも使える。
- 異なる長さのイテレータを与えると、zip は何のエラーも出さずに最短で停止する。
- 複数のイテレータの長さが異なるときに、最短で停止することなく zip するには、組み込みモジュール itertools の zip_longest 関数を使う。

## 補足

以下の文について

> zip はラップしているイテレータで要素を１つずつ処理します。したがって、無限に長い入力でもメモリを使いすぎてクラッシュする危険性はありません

かんたんに言うと、「zip は全部をまとめて一気に持ってこないで、1要素ずつ順番に取り出しながら処理する仕組みだよ、という話です。

もう少し丁寧に

例えば zip(a, b) と書いたとき、
a と b がリストなら全部の要素を持ってるのでそんなに意識しないんだけど、
実は zip は イテレータ にも使える。

イテレータって「必要なときにだけ値を1つずつ生み出す装置」みたいなもの。
（たとえば range(10**12) みたいに「実際の中身を全部メモリに持ってない」ものも含む）

zip は、内部でこんな感じの動きをする：

```
次の a の値を1つとる
次の b の値を1つとる
それらをタプル (a_i, b_i) にして返す
```

これを 繰り返すだけ。

つまり一度に全部をメモリに読み込まない。

例：無限に長いイテレータでも OK

Python には、たとえばこんな「無限カウンタ」を作れる：

```Python
import itertools

counter = itertools.count()  # 0, 1, 2, 3, ... と無限に出てくる
```

これと range(5) を zip すると：

```Python
for a, b in zip(counter, range(5)):
    print(a, b)
```

```sh
0 0
1 1
2 2
3 3
4 4
```

はい、ちゃんと終わる。

ここで重要なのは：

- counter は無限
- でも zip は毎回「1つずつだけ」取り出す
- だから「無限」でもメモリを無限に使わない

### クラッシュする危険性があるのは？

**「無限の値を実際にためこもうとしたとき」**です。

例1: print で永遠に出力し続ける場合

```Python
for x in itertools.count():
    print(x)
```

これは
メモリが先にクラッシュすることもあるし、
単に永遠に出力が続くだけで終わらない。

例2: リストにしてしまった場合（これはクラッシュする）

```Python
list(itertools.count())
```

これは 無限の値を全部メモリに保持しようとするので
メモリを食い尽くしてクラッシュする。


### まとめ

| 処理                            | メモリ安全か？    | 理由           |
| ----------------------------- | ---------- | ------------ |
| `counter = itertools.count()` | ✅ 安全       | 何も溜め込まない     |
| `for x in counter: print(x)`  | ✅ ただし終わらない | 溜めないが無限ループ   |
| `list(counter)`               | ❌ クラッシュ    | 無限を溜め込もうとする  |
| `zip(counter, range(5))`      | ✅ 安全       | 必要な分だけ1つずつ消費 |
